# Aula 02 · Aprendizagem e retropropagação

Vamos calcular a previsão da rede 2–2–1, medir a perda e descobrir como cada peso e viés influencia essa perda. Usaremos NumPy para acompanhar as contas e TensorFlow/Keras para conferir os gradientes e realizar um passo de SGD.

Execute as seções em ordem com o kernel da pasta `codigos`. O exemplo usa dados definidos no notebook e pode rodar na CPU.

## 01 · Propagação direta, logit e perda

A entrada é $x_1=1$ e $x_2=2$. O alvo é $y=1$. A rede tem **9 parâmetros**: quatro pesos e dois vieses na camada oculta e dois pesos e um viés na saída.

<img src="figuras/05_arquitetura_mlp.png" alt="Rede com duas entradas, dois neurônios ocultos e uma saída" width="720">

### Dicionário de símbolos

Em $w_{ij}$, o primeiro índice indica a entrada e o segundo indica o neurônio oculto.

| Símbolo | Como ler no exemplo |
|---|---|
| $x_i$ | Valor de entrada $i$ |
| $w_{ij}$ | Peso que liga a entrada $i$ ao neurônio oculto $j$ |
| $a_j$ | Soma ponderada com viés, antes da tanh, no neurônio $j$ |
| $h_j$ | Ativação do neurônio oculto $j$, depois da tanh |
| $v_j$ | Peso que liga $h_j$ à saída |
| $z$ | Logit, o escore bruto da rede |
| $p$, $y$, $L$ | Probabilidade da classe 1, alvo conhecido e perda |

### As contas da primeira previsão

Na fórmula, $b_j$ é o viés do neurônio oculto $j$ e $b_{\mathrm{saída}}$ é o viés da saída.

$$
a_j=\sum_i x_iw_{ij}+b_j,\qquad h_j=\tanh(a_j),\qquad
z=v_1h_1+v_2h_2+b_{\mathrm{saída}}.
$$

$$
\tanh(a)=\frac{e^{2a}-1}{e^{2a}+1},\qquad
p=\sigma(z)=\frac{1}{1+e^{-z}}.
$$

A primeira soma é $1(0{,}1)+2(0{,}4)+0=0{,}9$. A segunda é $1(-0{,}2)+2(0{,}3)+0{,}1=0{,}5$. Após a tanh, obtemos $h_1\approx0{,}71630$ e $h_2\approx0{,}46212$.

In [ ]:
import numpy as np

X = np.array([[1.0,
               2.0]], dtype=np.float64)

y = np.array([[1.0]], dtype=np.float64)

In [2]:
# Uma linha por origem e uma coluna por destino em cada matriz de pesos.
W1 = np.array([[0.1, -0.2], [0.4, 0.3]], dtype=np.float64) # pesos da camada de entrada para a camada oculta
b1 = np.array([0.0, 0.1], dtype=np.float64) # bias da camada oculta

W2 = np.array([[0.2], [-0.5]], dtype=np.float64) # pesos da camada oculta para a camada de saída
b2 = np.array([0.05], dtype=np.float64) # bias da camada de saída
parametros = (W1, b1, W2, b2)
nomes_parametros = ("W1", "b1", "W2", "b2")

np.set_printoptions(precision=8, suppress=True)

In [3]:
def propagacao_numpy(entradas, parametros_rede):
    pesos_oculta, vies_oculta, pesos_saida, vies_saida = parametros_rede

    # Cada linha de A1 e H corresponde a uma observação.
    A1 = entradas @ pesos_oculta + vies_oculta
    H = np.tanh(A1)
    z = H @ pesos_saida + vies_saida

    # sigmoid(z) = exp(-log(1 + exp(-z))), escrita de forma estável.
    p = np.exp(-np.logaddexp(0.0, -z))
    return A1, H, z, p

In [4]:
def perda_numpy(alvos, logits):
    # logaddexp(0, z) calcula ln(1 + exp(z)) de forma estável.
    return np.mean(np.logaddexp(0.0, logits) - alvos * logits)

In [5]:
A1, H, z, p = propagacao_numpy(X, parametros)
perda_antes_numpy = perda_numpy(y, z)

print("Somas com viés", A1)
print("Ativações", H)

Somas com viés [[0.9 0.5]]
Ativações [[0.71629787 0.46211716]]


In [6]:
print("Logit", z)
print("Probabilidade da classe 1", p)

Logit [[-0.037799]]
Probabilidade da classe 1 [[0.49055137]]


In [7]:
print(f"Perda inicial {perda_antes_numpy:.8f}")

Perda inicial 0.71222527


### Duas formas de calcular a entropia cruzada binária

$$
L(y,p)=-[y\ln(p)+(1-y)\ln(1-p)].
$$

Como o alvo é 1, a perda do exemplo é $-\ln(0{,}49055)\approx0{,}71223$. O cálculo direto com o logit chega à mesma perda:

$$
L(y,z)=\ln(1+e^z)-yz
\approx 0{,}674426-1(-0{,}037799)
\approx 0{,}712225.
$$

Usamos `logaddexp` para evitar calcular isoladamente exponenciais ou probabilidades extremas. Keras aplica uma forma estável a partir de `z` com `BinaryCrossentropy(from_logits=True)`.

In [11]:
# A probabilidade deste exemplo está longe de 0 e 1, então podemos comparar as duas formas.
perda_via_probabilidade = -np.mean(y * np.log(p) + (1 - y) * np.log1p(-p))
print(f"Perda pela probabilidade {perda_via_probabilidade:.10f}")
print(f"Perda pelo logit {perda_antes_numpy:.10f}")

Perda pela probabilidade 0.7122252678
Perda pelo logit 0.7122252678


## 02 · Retropropagação em NumPy e diferenças finitas

A derivada parcial mede como a perda muda perto do valor atual de um parâmetro, mantendo os outros fixos. As derivadas de todos os pesos e vieses formam o gradiente.

Na saída, usamos $\partial L/\partial z=p-y$ para um exemplo. O fator local de $v_1$ é $h_1$, o de $v_2$ é $h_2$ e o do viés é 1.

$$
\frac{\partial L}{\partial v_j}=\frac{\partial L}{\partial z}h_j,\qquad
\frac{\partial L}{\partial b_{\mathrm{saída}}}=\frac{\partial L}{\partial z}.
$$

No termo $v_1h_1$ do logit, $v_1$ fica fixo quando variamos $h_1$. Por isso, $\partial z/\partial h_1=v_1=0{,}2$ é o fator que usamos para voltar de $z$ a $h_1$. Já $\partial L/\partial v_1\approx-0{,}36492$ mede como a perda muda quando variamos o próprio peso $v_1$. Esse gradiente é usado no passo de atualização de $v_1$.

Ao voltar ao neurônio oculto $j$, multiplicamos pelo peso $v_j$ usado na ida e pela derivada da tanh, $\partial h_j/\partial a_j=1-h_j^2$.

$$
\frac{\partial L}{\partial a_j}=\frac{\partial L}{\partial z}v_j(1-h_j^2),
\qquad
\frac{\partial L}{\partial w_{ij}}=x_i\frac{\partial L}{\partial a_j}.
$$

Por exemplo, $\partial L/\partial w_{11}\approx (-0{,}50945)(0{,}2)(1-0{,}71630^2)(1)\approx -0{,}04961$. Esse sinal indica que aumentar um pouco $w_{11}$ reduz a perda, com os outros parâmetros fixos.

In [12]:
def gradientes_numpy(entradas, alvos, parametros_rede):
    _, H, _, p = propagacao_numpy(entradas, parametros_rede)
    _, _, pesos_saida, _ = parametros_rede

    # Um exemplo dá p - y. Com perda média, dividimos pelo tamanho do lote.
    dz = (p - alvos) / len(entradas)

    # A transposta distribui o sinal de saída pelos neurônios ocultos.
    # A multiplicação por 1 - H**2 aplica a derivada da tanh elemento a elemento.
    dA1 = (dz @ pesos_saida.T) * (1 - H**2)

    dW1 = entradas.T @ dA1
    db1 = dA1.sum(axis=0)
    dW2 = H.T @ dz
    db2 = dz.sum(axis=0)
    return dW1, db1, dW2, db2

In [13]:
gradientes_np = gradientes_numpy(X, y, parametros)
for nome, gradiente in zip(nomes_parametros, gradientes_np):
    print(f"Gradiente de {nome}\n{gradiente}")

Gradiente de W1
[[-0.04961188  0.20032736]
 [-0.09922375  0.40065472]]
Gradiente de b1
[-0.04961188  0.20032736]
Gradiente de W2
[[-0.36491697]
 [-0.23542495]]
Gradiente de b2
[-0.50944863]


### Um peso, duas perdas

$\varepsilon=10^{-6}$ é a distância entre o peso original e cada peso perturbado. Os dois pesos perturbados ficam a $2\varepsilon$ de distância entre si.

$$
\frac{\partial L}{\partial w_{11}}\approx
\frac{L(w_{11}+\varepsilon)-L(w_{11}-\varepsilon)}{2\varepsilon}.
$$

Cada $L$ vem de uma nova propagação direta. A diferença das perdas dividida pela diferença dos pesos aproxima a inclinação no peso original.

In [14]:
def perda_com_w11(w11):
    # Copiamos apenas a matriz que será perturbada. Os pesos da ida ficam intactos.
    W1_perturbado = W1.copy()
    W1_perturbado[0, 0] = w11
    _, _, z_perturbado, _ = propagacao_numpy(X, (W1_perturbado, b1, W2, b2))
    return perda_numpy(y, z_perturbado)

In [15]:
epsilon = 1e-6
w11 = W1[0, 0]
perda_mais = perda_com_w11(w11 + epsilon)
perda_menos = perda_com_w11(w11 - epsilon)
gradiente_finito = (perda_mais - perda_menos) / (2 * epsilon)

print(f"Peso menor {w11 - epsilon:.6f}  ·  perda {perda_menos:.12f}")
print(f"Peso maior {w11 + epsilon:.6f}  ·  perda {perda_mais:.12f}")

Peso menor 0.099999  ·  perda 0.712225317429
Peso maior 0.100001  ·  perda 0.712225218206


In [16]:
print(f"Inclinação {gradiente_finito:.8f}")
print(f"Gradiente pela retropropagação {gradientes_np[0][0, 0]:.8f}")

Inclinação -0.04961188
Gradiente pela retropropagação -0.04961188


## 03 · GradientTape e um passo de SGD com Keras

Montamos as duas camadas `Dense` do mesmo tamanho que as matrizes de NumPy e copiamos seus parâmetros. `GradientTape` acompanha a propagação direta e o cálculo da perda. Depois, `tape.gradient` obtém a derivada de cada variável treinável.

In [17]:
import tensorflow as tf
from tensorflow import keras

In [18]:
def criar_modelo(parametros_rede):
    # A saída Dense é linear. O Keras recebe z diretamente na BCE.
    modelo = keras.Sequential([
        keras.Input(shape=(2,), dtype="float64"),
        keras.layers.Dense(2, activation="tanh", dtype="float64"),
        keras.layers.Dense(1, dtype="float64"),
    ])
    modelo.set_weights(parametros_rede)
    return modelo

In [19]:
bce = keras.losses.BinaryCrossentropy(from_logits=True, dtype="float64")
X_tf = tf.constant(X)
y_tf = tf.constant(y)

In [21]:
modelo = criar_modelo(parametros)
with tf.GradientTape() as tape:
    z_keras = modelo(X_tf)
    perda_antes_keras = bce(y_tf, z_keras)

gradientes_keras = tape.gradient(perda_antes_keras, modelo.trainable_variables)

In [22]:
modelo.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 2)              │             6 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9 (72.00 B)

 Trainable params: 9 (72.00 B)

 Non-trainable params: 0 (0.00 B)

In [23]:
maior_diferenca = max(
    np.max(np.abs(grad_np - grad_keras.numpy()))
    for grad_np, grad_keras in zip(gradientes_np, gradientes_keras)
)
print("Parâmetros treináveis", modelo.count_params())
print(f"Maior diferença entre gradientes {maior_diferenca:.3e}")

Parâmetros treináveis 9
Maior diferença entre gradientes 5.551e-17


### Um passo de atualização

Para cada peso, usamos $w_{ij,\mathrm{novo}}=w_{ij,\mathrm{atual}}-\eta\,\partial L/\partial w_{ij}$. A mesma regra vale para os vieses. Com $\eta=0{,}1$, $w_{11}$ aumenta para aproximadamente $0{,}10496$ e $w_{12}$ diminui para aproximadamente $-0{,}22003$.

Guardamos os gradientes antes de mudar qualquer parâmetro. Depois do passo, uma nova propagação direta mede o efeito sobre o mesmo exemplo.

<img src="figuras/21_perda_antes_e_depois.png" alt="Rede após a atualização com novos pesos, probabilidade e perda" width="720">

In [26]:
print(parametros)
print(gradientes_np)

(array([[ 0.1, -0.2],
       [ 0.4,  0.3]]), array([0. , 0.1]), array([[ 0.2],
       [-0.5]]), array([0.05]))
(array([[-0.04961188,  0.20032736],
       [-0.09922375,  0.40065472]]), array([-0.04961188,  0.20032736]), array([[-0.36491697],
       [-0.23542495]]), array([-0.50944863]))


In [24]:
taxa = 0.1
parametros_novos = tuple(
    parametro - taxa * gradiente
    for parametro, gradiente in zip(parametros, gradientes_np)
)
_, _, z_novo, p_novo = propagacao_numpy(X, parametros_novos)

In [28]:
# Um modelo novo parte sempre dos pesos da ida, inclusive ao reexecutar a célula.
modelo_atualizado = criar_modelo(parametros)
sgd = keras.optimizers.SGD(learning_rate=taxa)
sgd.apply_gradients(zip(gradientes_keras, modelo_atualizado.trainable_variables))
z_novo_keras = modelo_atualizado(X_tf)
perda_depois_keras = bce(y_tf, z_novo_keras)
perda_depois_numpy = perda_numpy(y, z_novo)

In [29]:
print(f"Probabilidade {p.item():.5f} → {p_novo.item():.5f}")
print(f"Perda em NumPy {perda_antes_numpy:.5f} → {perda_depois_numpy:.5f}")
print(f"Perda em Keras {perda_antes_keras.numpy():.5f} → {perda_depois_keras.numpy():.5f}")

Probabilidade 0.49055 → 0.52522
Perda em NumPy 0.71223 → 0.64394
Perda em Keras 0.71223 → 0.64394


## 04 · Um exemplo ou um lote

Com `batch_size` igual a 1, cada exemplo gera um passo. Com $n$ exemplos no lote, fazemos a média das perdas e um passo usando o gradiente médio.

$$
L_{\mathrm{lote}}=\frac{1}{n}\sum_{k=1}^n L_k,\qquad
\nabla L_{\mathrm{lote}}=\frac{1}{n}\sum_{k=1}^n \nabla L_k.
$$

As duas maneiras de calcular a média usam o mesmo estado inicial da rede. Vamos conferir a igualdade com duas observações. A primeira é a do slide. A segunda é sintética e tem alvo 0.

Na GPU, o tamanho do lote afeta o espaço ocupado pelas ativações. O treino também guarda parâmetros, gradientes e, quando aplicável, o estado do otimizador.

In [ ]:
X_lote = np.array([[1.0, 2.0],
                   [2.0, 1.0]])
y_lote = np.array([[1.0], [0.0]])

In [ ]:
gradientes_lote = gradientes_numpy(X_lote, y_lote, parametros)

gradientes_individuais = [
    gradientes_numpy(X_lote[i:i + 1], y_lote[i:i + 1], parametros)
    for i in range(len(X_lote))
]

In [33]:
gradientes_lote

(array([[ 0.05194077, -0.16947203],
        [-0.01123852,  0.0655095 ]]),
 array([ 0.01356742, -0.03465418]),
 array([[-0.03765074],
        [-0.11771248]]),
 array([0.0149114]))

In [35]:
gradientes_individuais

[(array([[-0.04961188,  0.20032736],
         [-0.09922375,  0.40065472]]),
  array([-0.04961188,  0.20032736]),
  array([[-0.36491697],
         [-0.23542495]]),
  array([-0.50944863])),
 (array([[ 0.15349342, -0.53927142],
         [ 0.07674671, -0.26963571]]),
  array([ 0.07674671, -0.26963571]),
  array([[ 0.28961548],
         [-0.        ]]),
  array([0.53927142]))]

In [36]:
media_w11 = np.mean([gradientes[0][0, 0] for gradientes in gradientes_individuais])

print(f"Gradiente de w11 no lote {gradientes_lote[0][0, 0]:.8f}")
print(f"Média dos gradientes individuais de w11 {media_w11:.8f}")

Gradiente de w11 no lote 0.05194077
Média dos gradientes individuais de w11 0.05194077
